# Bahth بحث — image search in Arabic, Turkish and English

## Part 3: from a fine-tuned model to something that can be served

Part 2 produced a text encoder that roughly doubles first-hit accuracy for Arabic and Turkish queries. This notebook turns it into the files a search service needs:

1. export the text encoder to ONNX, quantise it to int8, and measure what each step costs in retrieval accuracy
2. embed about 25,000 Unsplash photos with the frozen CLIP image tower
3. run real queries through the exact code the service uses

Code: https://github.com/readaloush/bahth

**Settings:** Internet on, GPU accelerator.

In [ ]:
!pip -q install sentence-transformers onnx onnxruntime kagglehub
!git clone -q https://github.com/readaloush/bahth.git
%cd bahth
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 1. The fine-tuned text encoder

Taken from the output of the Part 2 notebook. If that output cannot be read, the selected recipe is trained again here (about 15 minutes).

In [ ]:
import os, subprocess
STUDENT = None
try:
    import kagglehub
    root = kagglehub.notebook_output_download('readaloush/bahth-2-fine-tuning-for-arabic-and-turkish-image')
    for d, _, files in os.walk(root):
        if 'modules.json' in files:
            STUDENT = d
            break
    print('student from Part 2 output:', STUDENT)
except Exception as e:
    print('Part 2 output not readable:', type(e).__name__, str(e)[:200])
if STUDENT is None:
    subprocess.run("python -m bahth.train --data data --steps 1200 --recipes both 2>&1 | grep -v -E 'it/s\\]|Warning|warn|B/s\\]|^$'", shell=True, check=True)
    STUDENT = 'results/student'
os.environ['STUDENT'] = STUDENT

## 2. Export and measure

The same queries are scored three times: PyTorch, ONNX float32, ONNX int8. The ONNX runs go through `bahth.search.TextEncoder`, the class the service uses, so tokenisation differences would show up here too.

In [ ]:
!python -m bahth.export --student "$STUDENT" --data data --out artifacts 2>&1 | grep -v -E 'it/s\]|Warning|warn|B/s\]|^$'

## 3. The photo gallery

Unsplash Lite, about 25,000 photos. Only embeddings, photo ids and photographer credits are kept; the photos themselves are not stored or redistributed.

In [ ]:
!python -m bahth.gallery --out artifacts 2>&1 | grep -v -E 'it/s\]|Warning|warn|B/s\]|^$'

## 4. Search

Free-text queries through the int8 model and the exact index. Photos are shown by linking to Unsplash.

In [ ]:
import time, html
from IPython.display import HTML, display
from bahth.search import Index, TextEncoder
enc = TextEncoder('artifacts/text.int8.onnx', 'artifacts/tokenizer.json')
index = Index.load('artifacts')
queries = ['a lighthouse at sunset', 'منارة عند غروب الشمس', 'gün batımında deniz feneri',
           'قطة نائمة على أريكة', 'kanepede uyuyan kedi', 'سوق مزدحم في الليل', 'karlı dağlarda yürüyen insanlar',
           'فنجان قهوة على طاولة خشبية', 'yağmurlu bir şehir sokağı', 'children playing football']
for q in queries:
    t0 = time.perf_counter(); v = enc([q])[0]; t1 = time.perf_counter(); hits = index.search(v, 5); t2 = time.perf_counter()
    imgs = ''.join(f'<a href="https://unsplash.com/photos/{h["id"]}" title="{html.escape(h["by"])}"><img src="{h["image"]}?w=200&h=140&fit=crop" style="height:120px;margin:2px;border-radius:4px"></a>' for h in hits)
    display(HTML(f'<div dir="auto" style="margin-top:10px"><b>{html.escape(q)}</b> <span style="color:#888">encode {1000*(t1-t0):.0f} ms, search {1000*(t2-t1):.1f} ms, top score {hits[0]["score"]:.3f}</span></div>{imgs}'))
print(len(index), 'photos in the gallery')

## 5. Keep the serving files

`text.int8.onnx`, `tokenizer.json`, `gallery.npy`, `gallery.json` and the measurements. Everything else (datasets, the float32 model) is removed from the notebook output.

In [ ]:
!mkdir -p /kaggle/working/artifacts && cp artifacts/text.int8.onnx artifacts/tokenizer.json artifacts/gallery.npy artifacts/gallery.json artifacts/export.json /kaggle/working/artifacts/
%cd /kaggle/working
!rm -rf /kaggle/working/bahth && ls -la /kaggle/working/artifacts && cat /kaggle/working/artifacts/export.json | head -c 1500